[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/062.%20The%20Picker%20Routing%20Problem/P62-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 62. The Picker Routing Problem

## Tier 1: Markov Decision Process Formulation

### Purpose and scope

This notebook represents a picker route as a deterministic Markov decision process (MDP) and uses value iteration to compute a policy. The example has a depot, a small set of item locations, and a matrix of travel distances. The computed route visits every required item location and returns to the depot.

The MDP's objective is to **maximize discounted reward**, where each move incurs a negative travel-distance reward and each newly collected item earns a collection reward. Consequently, the reported route distance is a useful route statistic, but it is not by itself the MDP objective: the collection reward and discount factor can affect which route the policy selects. A shortest-distance route is guaranteed only when the reward and discounting are formulated to represent that objective.

### Model assumptions

- The distance matrix gives the travel cost for every ordered pair of locations. Distances are deterministic and nonnegative; the matrix need not be symmetric.
- The picker starts at the depot. Each item is collected by visiting its designated location once, and the picker returns to the depot after collecting all items.
- An action moves directly from the current location to one uncollected item location. Once all items are collected, the only available action (unless already at the depot) is to return to the depot.
- Travel and collection rewards are received immediately. Future rewards are discounted by a factor \(\gamma\), where \(0 \leq \gamma < 1\).
- The model does not represent aisle geometry, congestion, picking times, capacity limits, stochastic travel, or multiple pickers. The distance matrix summarizes travel between locations.

### The solution sequence

1. Define a state using the current location and the set of items not yet collected.
2. List feasible next locations (actions) for each state.
3. Compute the deterministic next state and immediate reward for each action.
4. Apply value iteration to obtain an optimal value and policy for every represented state.
5. Follow the policy from the initial state, then report and verify the resulting route.

### What to examine

- The value function estimates the maximum expected discounted reward from each state; here, transitions and rewards are deterministic.
- The policy identifies an action with the highest Bellman value in each nonterminal state.
- The extracted route shows the policy's item-collection order and return to the depot.
- The convergence history records the largest value change in each iteration.
- The sensitivity analysis shows how the chosen discount factor affects this model's route and convergence.

### MDP definitions and Bellman equation

A state must preserve exactly the information needed to determine future actions, rewards, and transitions. Here, the picker's current location alone is insufficient: the set of items still to collect also matters. The state is therefore \(s=(i,U)\), where \(i\) is the current location and \(U\) is the set of uncollected item locations. This state representation satisfies the Markov property: once \((i,U)\) is known, the earlier route does not affect the future model.

Let \(A(i,U)\) be the available actions. If \(U\) is nonempty, actions are the reachable locations in \(U\); if \(U\) is empty and \(i\) is not the depot, the action is to return to the depot. At the depot with no items remaining, the state is terminal. For action \(a\), the next state is \(s'=(a,U\setminus\{a\})\) when \(a\in U\), and otherwise \(s'=(a,U)\). The immediate reward is

$$r((i,U),a)=-d_{ia}+R\,\mathbf{1}[a\in U],$$

where \(d_{ia}\) is the distance from location \(i\) to location \(a\), \(R\) is the collection reward, and \(\mathbf{1}[a\in U]\) equals 1 when action \(a\) collects an uncollected item and 0 otherwise. Thus, travel is penalized and each item earns its collection reward once.

The optimal value function and policy satisfy the Bellman optimality equation

$$V^*(i,U)=\max_{a\in A(i,U)}\left[r((i,U),a)+\gamma V^*(s')\right].$$

For the terminal state, \(V^*(i,U)=0\) when \(U=\varnothing\) and \(i\) is the depot. The maximum is used because the model maximizes reward. The policy \(\pi^*(i,U)\) selects an action attaining this maximum. In the code, these quantities are stored in `self.values` and `self.policy`; `calculate_reward`, `get_next_state`, and `get_available_actions` implement the corresponding model components.

Value iteration repeatedly applies the Bellman update to all states. Each update uses the values from the preceding iteration, making the sweep synchronous:

$$V_{k+1}(s)=\max_{a\in A(s)}\left[r(s,a)+\gamma V_k(s')\right],$$

with terminal values fixed at zero. For a finite state space, bounded rewards, and \(0\leq\gamma<1\), the Bellman operator is a contraction, so value iteration converges to the unique optimal value function. The implementation stops when the largest absolute value change is at most the specified tolerance, or when it reaches the iteration limit.

With \(n\) locations and \(m\) item locations, the full state representation contains at most \(n2^m\) location-and-subset pairs. A sweep considers up to \(m\) actions per state, giving a straightforward worst-case cost of \(O(nm2^m)\) per iteration and \(O(n2^m)\) storage. The exponential dependence on the number of items limits the size of instances that can be handled this way.

In [ ]:
from itertools import combinations
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# A fixed seed keeps any later NumPy-based sampling reproducible.
np.random.seed(42)

print("Libraries imported successfully for the MDP formulation.")

In [ ]:
class PickerRoutingMDP:
    """MDP model for deterministic picker routing with value iteration."""

    def __init__(
        self,
        distance_matrix: np.ndarray,
        items: List[int],
        depot: int = 0,
        gamma: float = 0.95,
        collection_reward: float = 10.0,
    ):
        """Store the model data and check that the state space is well-defined.

        Args:
            distance_matrix: Nonnegative travel distances between locations.
            items: Locations at which required items are collected.
            depot: Location where the route starts and ends.
            gamma: Discount factor, required to be in [0, 1).
            collection_reward: Reward received when an uncollected item is visited.
        """
        distance_matrix = np.asarray(distance_matrix, dtype=float)
        if distance_matrix.ndim != 2 or distance_matrix.shape[0] != distance_matrix.shape[1]:
            raise ValueError("The distance matrix must be square.")
        if not np.all(np.isfinite(distance_matrix)) or np.any(distance_matrix < 0):
            raise ValueError("Distances must be finite and nonnegative.")

        self.n_locations = distance_matrix.shape[0]
        if not 0 <= depot < self.n_locations:
            raise ValueError("The depot index must identify a location in the matrix.")
        if not 0.0 <= gamma < 1.0:
            raise ValueError("The discount factor must satisfy 0 <= gamma < 1.")
        if not np.isfinite(collection_reward):
            raise ValueError("The collection reward must be finite.")

        self.items = set(items)
        if len(self.items) != len(items):
            raise ValueError("Each item location must appear only once in the input list.")
        if any(not isinstance(item, (int, np.integer)) for item in self.items):
            raise ValueError("Item locations must be integer indices.")
        if any(item < 0 or item >= self.n_locations for item in self.items):
            raise ValueError("Every item location must be in the distance matrix.")
        if depot in self.items:
            raise ValueError("The depot cannot also be an item location in this model.")

        self.distance_matrix = distance_matrix
        self.depot = depot
        self.gamma = gamma
        self.collection_reward = float(collection_reward)
        self.values: Dict[Tuple[int, frozenset], float] = {}
        self.policy: Dict[Tuple[int, frozenset], int] = {}
        self.convergence_history: List[float] = []

    def get_state_key(
        self, location: int, uncollected_items: frozenset
    ) -> Tuple[int, frozenset]:
        """Return a hashable key for a location-and-items state."""
        return (location, frozenset(uncollected_items))

    def get_available_actions(
        self, location: int, uncollected_items: frozenset
    ) -> List[int]:
        """List feasible next locations under the model's collection rules."""
        if uncollected_items:
            # Restrict movement to required, uncollected item locations.
            return [
                item
                for item in sorted(uncollected_items)
                if self.distance_matrix[location, item] > 0
            ]

        if location != self.depot:
            return [self.depot]

        return []

    def calculate_reward(
        self,
        current_location: int,
        action: int,
        uncollected_items: frozenset,
    ) -> float:
        """Return the immediate negative travel cost plus any collection reward."""
        travel_reward = -self.distance_matrix[current_location, action]
        collection_bonus = (
            self.collection_reward if action in uncollected_items else 0.0
        )
        return float(travel_reward + collection_bonus)

    def get_next_state(
        self,
        current_location: int,
        action: int,
        uncollected_items: frozenset,
    ) -> Tuple[int, frozenset]:
        """Apply the deterministic move and remove an item when it is collected."""
        next_uncollected = set(uncollected_items)
        next_uncollected.discard(action)
        return (action, frozenset(next_uncollected))

    def initialize_values(self) -> None:
        """Create a zero-valued entry for every location-and-subset state."""
        self.values.clear()
        self.policy.clear()
        self.convergence_history.clear()

        all_items = sorted(self.items)
        for subset_size in range(len(all_items) + 1):
            for item_subset in combinations(all_items, subset_size):
                uncollected_items = frozenset(item_subset)
                for location in range(self.n_locations):
                    state_key = self.get_state_key(location, uncollected_items)
                    self.values[state_key] = 0.0
                    self.policy[state_key] = None

    def value_iteration(
        self, max_iterations: int = 1000, tolerance: float = 1e-6
    ) -> int:
        """Compute the optimal values and policy with synchronous Bellman updates.

        Args:
            max_iterations: Maximum number of full state-space sweeps.
            tolerance: Stop when the largest value change is no greater than this.

        Returns:
            Number of sweeps performed.
        """
        if not self.values:
            raise RuntimeError("Initialize the state values before value iteration.")
        if max_iterations <= 0:
            raise ValueError("max_iterations must be positive.")
        if tolerance <= 0:
            raise ValueError("tolerance must be positive.")

        self.convergence_history.clear()
        state_keys = list(self.values)
        max_diff = float("inf")
        iteration = 0

        print(f"Starting value iteration with {len(state_keys)} states...")

        while iteration < max_iterations and max_diff > tolerance:
            old_values = self.values.copy()
            new_values = {}
            new_policy = {}
            max_diff = 0.0

            for state_key in state_keys:
                location, uncollected_items = state_key
                actions = self.get_available_actions(location, uncollected_items)

                if not actions:
                    new_values[state_key] = 0.0
                    new_policy[state_key] = None
                    continue

                action_values = []
                for action in actions:
                    reward = self.calculate_reward(
                        location, action, uncollected_items
                    )
                    next_state = self.get_next_state(
                        location, action, uncollected_items
                    )
                    action_value = reward + self.gamma * old_values[next_state]
                    action_values.append(action_value)

                best_value = max(action_values)
                best_action_index = next(
                    index
                    for index, value in enumerate(action_values)
                    if np.isclose(value, best_value, rtol=1e-12, atol=1e-12)
                )
                new_values[state_key] = best_value
                new_policy[state_key] = actions[best_action_index]
                max_diff = max(
                    max_diff, abs(best_value - old_values[state_key])
                )

            self.values = new_values
            self.policy = new_policy
            self.convergence_history.append(max_diff)
            iteration += 1

            if iteration % 10 == 0:
                print(f"Iteration {iteration}: maximum value change = {max_diff:.6g}")

        if max_diff <= tolerance:
            print(f"Value iteration converged in {iteration} iterations.")
        else:
            print(f"Value iteration stopped after {iteration} iterations without meeting the tolerance.")

        return iteration

    def extract_optimal_route(self) -> List[int]:
        """Follow the computed policy from the initial state to the terminal state."""
        route = []
        current_location = self.depot
        uncollected_items = frozenset(self.items)

        # At most one move per item, followed by the return trip and termination.
        for _ in range(len(self.items) + 2):
            route.append(current_location)
            if not uncollected_items and current_location == self.depot:
                return route

            state_key = self.get_state_key(current_location, uncollected_items)
            next_location = self.policy.get(state_key)
            if next_location is None:
                raise RuntimeError(f"No policy action is available for state {state_key}.")

            current_location, uncollected_items = self.get_next_state(
                current_location, next_location, uncollected_items
            )

        raise RuntimeError("The policy did not reach the terminal state within the expected route length.")

    def calculate_route_distance(self, route: List[int]) -> float:
        """Sum the matrix distances along a route."""
        return float(
            sum(
                self.distance_matrix[from_location, to_location]
                for from_location, to_location in zip(route[:-1], route[1:])
            )
        )

    def analyze_solution(self) -> Dict:
        """Summarize the extracted route, convergence, and value-function statistics."""
        if not self.values:
            raise RuntimeError("Initialize and solve the MDP before analyzing it.")

        optimal_route = self.extract_optimal_route()
        all_values = list(self.values.values())
        return {
            "optimal_route": optimal_route,
            "total_distance": self.calculate_route_distance(optimal_route),
            "num_states": len(self.values),
            "convergence_iterations": len(self.convergence_history),
            "final_convergence_error": (
                self.convergence_history[-1] if self.convergence_history else 0.0
            ),
            "value_function_stats": {
                "min": float(np.min(all_values)),
                "max": float(np.max(all_values)),
                "mean": float(np.mean(all_values)),
                "std": float(np.std(all_values)),
            },
        }

print("PickerRoutingMDP class defined successfully.")

## Concrete warehouse example

The following example has four locations: depot 0 and item locations 1, 2, and 3. Matrix entry \(d_{ij}\) is the distance from location \(i\) to location \(j\). The state space includes every location paired with every subset of uncollected items, including states that are not reached from the initial state. This gives \(4\times 2^3=32\) represented states.

The route \(0\to1\to2\to3\to0\) has distance \(5+4+3+6=18\) in this matrix. That arithmetic does not establish that it is the policy's route: the MDP maximizes discounted travel and collection rewards, so the policy is determined by the Bellman equation and the specified parameters.

In [ ]:
# The matrix entries are distances between the four warehouse locations.
distance_matrix = np.array(
    [
        [0, 5, 8, 6],
        [5, 0, 4, 7],
        [8, 4, 0, 3],
        [6, 7, 3, 0],
    ],
    dtype=float,
)

items = [1, 2, 3]
depot = 0

print("=== Picker Routing Problem: MDP Formulation ===")
print("\nDistance matrix (distance units):")
print(distance_matrix)
print(f"\nItem locations: {items}")
print(f"Depot location: {depot}")
print(f"Starting state: ({depot}, {set(items)})")

mdp = PickerRoutingMDP(
    distance_matrix=distance_matrix,
    items=items,
    depot=depot,
    gamma=0.95,
    collection_reward=10.0,
)

mdp.initialize_values()
print(f"\nInitialized {len(mdp.values)} states.")

print("\n=== Value iteration ===")
iterations = mdp.value_iteration(max_iterations=1000, tolerance=1e-6)

## Solution analysis and verification

The policy is evaluated from the initial state \((0,\{1,2,3\})\). The route check verifies that it starts and ends at the depot and visits each required item location exactly once. The reported distance is recomputed from the same matrix used by the MDP. It is a route measure, distinct from the discounted reward optimized by value iteration.

In [ ]:
analysis = mdp.analyze_solution()
route = analysis["optimal_route"]

visited_items = [location for location in route[1:-1] if location in mdp.items]
assert route[0] == depot and route[-1] == depot
assert len(visited_items) == len(items)
assert set(visited_items) == set(items)
assert len(visited_items) == len(set(visited_items))

print("=== Solution analysis ===")
print(f"Policy route: {route}")
print(f"Total route distance: {analysis['total_distance']:.2f} distance units")
print(f"Number of represented states: {analysis['num_states']}")
print(f"Value-iteration sweeps: {analysis['convergence_iterations']}")
print(f"Final maximum value change: {analysis['final_convergence_error']:.8g}")
print("Route verification: depot and item-visit requirements are satisfied.")

print("\n=== Value-function statistics across represented states ===")
for statistic, value in analysis["value_function_stats"].items():
    print(f"{statistic.capitalize()}: {value:.4f}")

## Reading state values and policy actions

A value belongs to a complete state, not just a warehouse location. For example, the same location can have different values depending on which items remain. A policy action is the next location selected by the Bellman maximization; `None` indicates a terminal state with no available action.

In [ ]:
print("=== Sample state values and policy actions ===")
print("State (location, uncollected items) -> value, next location")
print("-" * 65)

key_states = [
    (0, frozenset({1, 2, 3})),
    (1, frozenset({2, 3})),
    (2, frozenset({3})),
    (3, frozenset()),
    (0, frozenset()),
]

for location, uncollected in key_states:
    state_key = mdp.get_state_key(location, uncollected)
    value = mdp.values[state_key]
    policy_action = mdp.policy[state_key]
    print(
        f"({location}, {set(uncollected)}) -> "
        f"{value:.4f}, {policy_action}"
    )

## Visualizing the MDP results

The panels show value-iteration convergence, the sequence of location indices in the selected route, the distance matrix, and the distribution of computed state values. The route panel is a sequence diagram, not a physical warehouse map: the matrix supplies pairwise distances but no spatial coordinates.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle("Picker Routing MDP Analysis", fontsize=16, fontweight="bold")

# Each point records the largest value change in one full Bellman sweep.
ax1 = axes[0, 0]
if mdp.convergence_history:
    ax1.plot(
        range(1, len(mdp.convergence_history) + 1),
        mdp.convergence_history,
        marker="o",
    )
    ax1.set_yscale("symlog", linthresh=1e-12)
ax1.set_xlabel("Value-iteration sweep")
ax1.set_ylabel("Maximum absolute value change")
ax1.set_title("Value-iteration convergence")
ax1.grid(True, alpha=0.3)

# Location indices are shown against route step; no physical coordinates are assumed.
ax2 = axes[0, 1]
route = analysis["optimal_route"]
route_steps = range(len(route))
ax2.plot(route_steps, route, "o-", linewidth=2, label="Policy route")
for step, location in enumerate(route):
    ax2.annotate(f"Location {location}", (step, location), xytext=(4, 6), textcoords="offset points")
ax2.set_xlabel("Route step")
ax2.set_ylabel("Location index")
ax2.set_title("Route as a sequence of locations")
ax2.set_xticks(list(route_steps))
ax2.grid(True, alpha=0.3)
ax2.legend()

ax3 = axes[1, 0]
sns.heatmap(
    distance_matrix,
    annot=True,
    fmt=".0f",
    cmap="YlOrRd",
    ax=ax3,
    xticklabels=[f"Location {i}" for i in range(len(distance_matrix))],
    yticklabels=[f"Location {i}" for i in range(len(distance_matrix))],
    cbar_kws={"label": "Distance units"},
)
ax3.set_title("Distance matrix")
ax3.set_xlabel("To location")
ax3.set_ylabel("From location")

ax4 = axes[1, 1]
all_values = list(mdp.values.values())
ax4.hist(all_values, bins=20, alpha=0.7, color="skyblue", edgecolor="black")
ax4.set_xlabel("State value, V(s)")
ax4.set_ylabel("Number of states")
ax4.set_title("Value-function distribution")
ax4.grid(True, alpha=0.3)
ax4.axvline(x=0, color="red", linestyle="--", alpha=0.7, label="Zero value")
ax4.legend()

fig.tight_layout()
plt.show()

## Sensitivity analysis: discount factor

The discount factor controls how strongly future rewards are weighted relative to immediate rewards. A larger \(\gamma\) gives future collection rewards more influence relative to rewards received sooner. Because the reward includes both travel penalties and collection bonuses, changing \(\gamma\) can affect the policy route as well as state values. This experiment keeps the distance matrix, item locations, depot, and collection reward fixed while solving the same MDP for each listed discount factor.

In [ ]:
discount_factors = [0.5, 0.7, 0.9, 0.95, 0.99]
results = []

for gamma in discount_factors:
    mdp_test = PickerRoutingMDP(
        distance_matrix=distance_matrix,
        items=items,
        depot=depot,
        gamma=gamma,
        collection_reward=10.0,
    )
    mdp_test.initialize_values()
    mdp_test.value_iteration(max_iterations=1000, tolerance=1e-6)
    analysis_test = mdp_test.analyze_solution()

    results.append(
        {
            "gamma": gamma,
            "total_distance": analysis_test["total_distance"],
            "convergence_iterations": analysis_test["convergence_iterations"],
            "route": analysis_test["optimal_route"],
        }
    )

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

gammas = [result["gamma"] for result in results]
distances = [result["total_distance"] for result in results]
ax1.plot(gammas, distances, "bo-", linewidth=2, markersize=8)
ax1.set_xlabel("Discount factor, gamma")
ax1.set_ylabel("Policy-route distance (distance units)")
ax1.set_title("Policy-route distance by discount factor")
ax1.set_xticks(gammas)
ax1.grid(True, alpha=0.3)

iteration_counts = [result["convergence_iterations"] for result in results]
ax2.plot(gammas, iteration_counts, "ro-", linewidth=2, markersize=8)
ax2.set_xlabel("Discount factor, gamma")
ax2.set_ylabel("Value-iteration sweeps")
ax2.set_title("Convergence sweeps by discount factor")
ax2.set_xticks(gammas)
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print("=== Discount-factor sensitivity results ===")
print("Gamma | Route distance | Sweeps | Policy route")
print("-" * 65)
for result in results:
    print(
        f"{result['gamma']:.2f} | "
        f"{result['total_distance']:.2f} | "
        f"{result['convergence_iterations']:6d} | "
        f"{result['route']}"
    )

## Strengths, limitations, and interpretation

### Strengths

- **State-by-state policy:** The method computes a value and a policy action for every represented state, rather than reporting only one route.
- **Explicit sequential decisions:** The state records the current location and remaining items, so the effect of each visit on future choices is represented directly.
- **Parameter analysis:** The same state and Bellman-update structure can be solved for different discount factors and other model inputs.

### Limitations

- **Exponential state growth:** The number of represented states grows as \(n2^m\), where \(n\) is the number of locations and \(m\) is the number of items.
- **Reward-objective dependence:** The policy maximizes discounted reward, not necessarily unweighted total distance. Collection rewards and discounting can change the selected route.
- **Simplified operations:** The model omits aisle-level movement, congestion, time windows, picking duration, capacity, uncertainty, and interactions between pickers.

Interpret the value function in the units and objective induced by the reward definition, not as a distance. Interpret the route distance as the sum of matrix entries along the selected policy route. The two measures answer different questions under this discounted-reward formulation. Larger instances can be explored by extending the same model with additional operational constraints or data, while recognizing that the state-space growth remains exponential in the number of items.